# Priority 1 — ADBench Real Dataset Benchmarks (v2 — NaN fixed)

Evaluates **Deep-MELU** against 4 baselines on 8 real-world ADBench datasets.

**v2 changes vs v1:**
- Fixed NaN crash on low-dim datasets (wrong He init scale, now per-layer)
- Added `safe_cholesky_inv()` with adaptive regularisation
- Added NaN guard after whitening
- Added ADBench install check

**Run order:** Cell 1 (install check) → Cell 2 (imports) → Cell 3 (model) → ... → Cell 9 (summary)

## Cell 1 — Install check

Run this first. If ADBench is not installed the notebook uses a synthetic fallback that is too easy — all methods score 1.0. **Install ADBench to get meaningful results.**

In [ ]:
import subprocess, sys

def check_install(pkg, install_name=None):
    try:
        __import__(pkg)
        print(f"  {pkg:<20} ✓ installed")
        return True
    except ImportError:
        install_name = install_name or pkg
        print(f"  {pkg:<20} ✗ not found  →  pip install {install_name}")
        return False

print("Dependency check:")
print("-" * 45)
adbench_ok  = check_install("adbench")
sklearn_ok  = check_install("sklearn", "scikit-learn")
scipy_ok    = check_install("scipy")
matplotlib_ok = check_install("matplotlib")
seaborn_ok  = check_install("seaborn")
posthoc_ok  = check_install("scikit_posthocs", "scikit-posthocs")
print("-" * 45)

if not adbench_ok:
    print()
    print("  ⚠️  ADBench not installed.")
    print("  The notebook will run with synthetic fallback data.")
    print("  Fallback data is too easy — all methods score ~1.0.")
    print("  For paper-quality results, install ADBench:")
    print()
    print("      pip install adbench")
    print()
    print("  Then restart the kernel and rerun.")
else:
    print()
    print("  ✓ ADBench installed — real dataset results will be used.")


## Cell 2 — Imports

In [ ]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import betainc
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score
)
warnings.filterwarnings("ignore")
np.random.seed(42)
print("All imports OK")


## Cell 3 — Deep-MELU (v2 — NaN fixed)

**Root cause of previous NaN:** W2 was initialised with `sqrt(2/input_dim)` instead of `sqrt(2/hidden_dim)`. This made W2 values ~3.3x too large for low-dim inputs (dim=6), causing latent activations to range up to 100+, making the MCD covariance matrix poorly conditioned, and APN-MCD whitening to produce NaN.

**Fix:** proper He initialisation per layer + `safe_cholesky_inv()` + NaN guard.

In [ ]:
def _tcdf(x, nu=4.0):
    nu = max(float(nu), 2.0)
    z  = nu / (nu + np.clip(x**2, 1e-30, None))
    ib = betainc(nu/2, 0.5, np.clip(z, 1e-12, 1-1e-12))
    return np.where(x >= 0, 1.0 - ib/2.0, ib/2.0)


def safe_cholesky_inv(cov, latent, n_samples):
    """
    Robust Cholesky + inverse.
    Adaptively scales regularisation epsilon based on:
      - the covariance diagonal magnitude
      - the ratio of samples to latent dimensions
    Falls back to identity if all attempts fail.
    """
    diag_max = max(np.diag(cov).max(), 1e-8)
    sample_ratio = n_samples / max(latent, 1)
    # More regularisation when few samples per dimension
    base_eps = diag_max * max(1e-3, min(0.1, 5.0 / max(sample_ratio, 1)))

    for eps_try in [base_eps, base_eps*10, base_eps*100, diag_max*0.5]:
        try:
            L  = np.linalg.cholesky(cov + eps_try * np.eye(latent))
            Li = np.linalg.inv(L)
            if np.isnan(Li).any() or np.isinf(Li).any():
                continue
            if np.linalg.cond(Li) > 1e7:
                continue
            return Li
        except np.linalg.LinAlgError:
            continue
    return np.eye(latent)   # safe fallback: Euclidean whitening


class DeepMELU:
    """
    Numpy simulation of Deep-MELU. v2 — NaN fixes applied.
    """
    def __init__(self, dim, hidden=64, latent=32,
                 alpha=1.0, beta=0.4, nu=4.0, momentum=0.95):
        self.dim = dim; self.latent = latent
        self.alpha = alpha; self.beta = beta
        self.nu = nu; self.momentum = momentum
        np.random.seed(0)
        # FIX: proper He init per layer (was: all using sqrt(2/dim))
        self.W1 = np.random.randn(dim,    hidden) * np.sqrt(2 / dim)
        self.W2 = np.random.randn(hidden, latent) * np.sqrt(2 / hidden)  # ← fixed
        self.Wd = np.random.randn(latent, dim)    * np.sqrt(2 / latent)  # ← fixed
        self.mu  = np.zeros(latent)
        self.Li  = np.eye(latent)
        self.tau = 1.0
        self.mu_d=0.; self.sd_d=1.; self.mu_e=0.; self.sd_e=1.; self.thr=1.

    def _sw(self, x):
        return x / (1 + np.exp(-np.clip(x, -50, 50)))

    def _enc(self, X):
        return self._sw(self._sw(X @ self.W1) @ self.W2)

    def _melu(self, H):
        T1  = H * _tcdf(H, self.nu)
        c   = H - self.mu
        w   = c @ self.Li.T
        # FIX: NaN guard on whitened values
        w   = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
        m   = np.sqrt(np.maximum((w**2).sum(1), 0))
        gate = (m >= self.tau).astype(float)[:, None]
        exp_arg = np.clip(self.beta * (m[:, None] - self.tau), -20, 20)
        amp  = self.alpha * np.sign(H) * (np.exp(exp_arg) - 1)
        return T1 + gate * amp

    def _mcd(self, Z, h_frac=0.75):
        n = len(Z); h = max(int(n * h_frac), self.latent + 1)
        best_det, best_mu, best_cov = np.inf, None, None
        for _ in range(8):
            idx = np.random.choice(n, h, replace=False); sub = Z[idx]
            for _ in range(6):
                mu  = sub.mean(0); d = sub - mu
                cov = d.T @ d / max(len(sub)-1, 1) + 1e-4*np.eye(self.latent)
                Si  = np.linalg.inv(cov)
                ds  = np.sqrt(np.maximum(
                    np.einsum('bi,ij,bj->b', Z-mu, Si, Z-mu), 0))
                idx = np.argsort(ds)[:h]; sub = Z[idx]
            mu = sub.mean(0); d = sub - mu
            cov = d.T @ d / max(len(sub)-1, 1)
            det = np.linalg.det(cov + 1e-4*np.eye(self.latent))
            if det < best_det:
                best_det = det; best_mu = mu; best_cov = cov
        return best_mu, best_cov

    def fit(self, X_train, n_epochs=60, lr=0.005, batch=64):
        n = len(X_train)
        for ep in range(n_epochs):
            Z = self._enc(X_train)
            # FIX: NaN guard on encoded Z
            Z = np.nan_to_num(Z, nan=0.0, posinf=1.0, neginf=-1.0)

            mu, cov = self._mcd(Z)
            self.mu  = mu
            self.Li  = safe_cholesky_inv(cov, self.latent, len(Z))

            c  = Z - mu; w = c @ self.Li.T
            w  = np.nan_to_num(w, nan=0.0)
            dm = np.sqrt(np.maximum((w**2).sum(1), 0))
            self.tau = self.momentum*self.tau + (1-self.momentum)*dm.mean()

            # Decoder SGD
            idx = np.random.permutation(n)
            for i in range(0, n, batch):
                xb  = X_train[idx[i:i+batch]]
                zb  = self._enc(xb)
                zb  = np.nan_to_num(zb, nan=0.0)
                zm  = self._melu(zb)
                zm  = np.nan_to_num(zm, nan=0.0)
                xh  = zm @ self.Wd
                err = xh - xb
                g   = np.clip(zm.T @ err / max(len(xb), 1), -1, 1)
                self.Wd -= lr * g

        # Calibrate
        Z   = np.nan_to_num(self._enc(X_train), nan=0.0)
        Zm  = np.nan_to_num(self._melu(Z), nan=0.0)
        Xh  = Zm @ self.Wd
        c   = Z - self.mu; w = np.nan_to_num(c @ self.Li.T, nan=0.0)
        dm  = np.sqrt(np.maximum((w**2).sum(1), 0))
        er  = np.abs(X_train - Xh).mean(1)
        self.mu_d = dm.mean(); self.sd_d = max(dm.std(), 1e-6)
        self.mu_e = er.mean(); self.sd_e = max(er.std(), 1e-6)
        sd  = np.maximum(0, (dm - self.mu_d) / self.sd_d)
        se  = np.maximum(0, (er - self.mu_e) / self.sd_e)
        self.thr = np.percentile(0.5*sd + 0.5*se, 95)
        return self

    def score(self, X):
        Z   = np.nan_to_num(self._enc(X), nan=0.0)
        Zm  = np.nan_to_num(self._melu(Z), nan=0.0)
        Xh  = Zm @ self.Wd
        c   = Z - self.mu; w = np.nan_to_num(c @ self.Li.T, nan=0.0)
        dm  = np.sqrt(np.maximum((w**2).sum(1), 0))
        er  = np.abs(X - Xh).mean(1)
        sd  = np.maximum(0, (dm - self.mu_d) / self.sd_d)
        se  = np.maximum(0, (er - self.mu_e) / self.sd_e)
        return 0.5*sd + 0.5*se


# Quick NaN regression test
np.random.seed(1)
X_test = np.random.randn(200, 6)   # low-dim, matches Thyroid
m_test  = DeepMELU(6)
m_test.fit(X_test, n_epochs=5)
sc = m_test.score(X_test)
has_nan = np.isnan(sc).any()
print(f"NaN regression test (dim=6): {'PASS ✓' if not has_nan else 'FAIL ✗'}")
print(f"Score range: [{sc.min():.4f}, {sc.max():.4f}]")


## Cell 4 — Metrics helper

In [ ]:
def evaluate(y_true, scores, thr_pct=95):
    if len(np.unique(y_true)) < 2:
        return dict(auroc=0.5, aucpr=0.0, f1=0.0, precision=0.0, recall=0.0)
    thr   = np.percentile(scores, thr_pct)
    y_hat = (scores > thr).astype(int)
    return dict(
        auroc     = float(roc_auc_score(y_true, scores)),
        aucpr     = float(average_precision_score(y_true, scores)),
        f1        = float(f1_score(y_true, y_hat, zero_division=0)),
        precision = float(precision_score(y_true, y_hat, zero_division=0)),
        recall    = float(recall_score(y_true, y_hat, zero_division=0)),
    )
print("evaluate() defined")


## Cell 5 — Dataset loader

In [ ]:
def load_dataset(name):
    try:
        from adbench.datasets.base import DataGenerator
        gen  = DataGenerator(dataset=name, seed=42)
        data = gen.generator(la=0)
        X, y = data['X_train'], data['y_train']
        print(f"  [{name}] ADBench  n={len(X)} dim={X.shape[1]} "
              f"outliers={int(y.sum())} ({y.mean()*100:.1f}%)")
        return X.astype(float), y.astype(int)
    except Exception as e:
        print(f"  [{name}] fallback (ADBench unavailable: {str(e)[:40]})")
        return _fallback(name)

def _fallback(name):
    SPECS = {
        "Thyroid":          dict(n=3772, dim=6,   cont=0.025),
        "WBC":              dict(n=378,  dim=30,  cont=0.056),
        "Annthyroid":       dict(n=7200, dim=6,   cont=0.074),
        "Lympho":           dict(n=148,  dim=18,  cont=0.041),
        "Cardiotocography": dict(n=2126, dim=21,  cont=0.096),
        "Ionosphere":       dict(n=351,  dim=33,  cont=0.359),
        "Arrhythmia":       dict(n=452,  dim=274, cont=0.150),
        "Satellite":        dict(n=6435, dim=36,  cont=0.316),
    }
    spec = SPECS.get(name, dict(n=500, dim=10, cont=0.10))
    n, d, c = spec["n"], spec["dim"], spec["cont"]
    n_out = max(1, int(n*c)); n_in = n - n_out
    np.random.seed(42)
    cov = np.eye(d) + 0.3*(np.random.randn(d,d)@np.random.randn(d,d).T)/(d*2)
    cov = (cov+cov.T)/2 + d*np.eye(d)*0.01
    X_in  = np.random.multivariate_normal(np.zeros(d), cov/d, n_in)
    X_out = np.random.randn(n_out,d)*2.5 + np.random.choice([-1,1],d)*3.0
    X = np.vstack([X_in, X_out])
    y = np.array([0]*n_in + [1]*n_out)
    print(f"  [{name}] fallback simulated  n={n} dim={d} outliers={n_out} ({c*100:.1f}%)")
    return X, y

print("load_dataset() defined")


## Cell 6 — Baseline methods

In [ ]:
class _VanillaAE:
    def __init__(self, dim, hid=64, lat=32):
        np.random.seed(1)
        # Proper He init per layer
        self.W1 = np.random.randn(dim, hid) * np.sqrt(2/dim)
        self.W2 = np.random.randn(hid, lat) * np.sqrt(2/hid)
        self.W3 = np.random.randn(lat, hid) * np.sqrt(2/lat)
        self.W4 = np.random.randn(hid, dim) * np.sqrt(2/hid)
    def _sw(self,x): return x/(1+np.exp(-np.clip(x,-50,50)))
    def _enc(self,X): return self._sw(self._sw(X@self.W1)@self.W2)
    def _dec(self,Z): return self._sw(Z@self.W3)@self.W4
    def fit(self,X,n_epochs=60,lr=0.003,batch=64):
        for _ in range(n_epochs):
            idx = np.random.permutation(len(X))
            for i in range(0,len(X),batch):
                xb=X[idx[i:i+batch]]; zb=self._enc(xb)
                h2=self._sw(zb@self.W3); xh=h2@self.W4; err=xh-xb
                g = np.clip(h2.T@err/max(len(xb),1), -1, 1)
                self.W4 -= lr * g
        return self
    def score(self,X): return ((X-self._dec(self._enc(X)))**2).mean(1)


def run_method(name, X_all, y):
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_all)
    X_in   = X_sc[y == 0]
    t0 = time.time()
    try:
        if name == "Deep-MELU":
            m = DeepMELU(X_all.shape[1])
            m.fit(X_in, n_epochs=60)
            scores = m.score(X_sc)
        elif name == "Isolation Forest":
            m = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
            m.fit(X_in); scores = -m.score_samples(X_sc)
        elif name == "LOF":
            m = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination="auto")
            m.fit(X_in); scores = -m.score_samples(X_sc)
        elif name == "OC-SVM":
            m = OneClassSVM(kernel="rbf", nu=0.1, gamma="scale")
            m.fit(X_in); scores = -m.score_samples(X_sc)
        elif name == "Vanilla AE":
            m = _VanillaAE(X_all.shape[1])
            m.fit(X_in); scores = m.score(X_sc)
        if np.isnan(scores).any():
            raise ValueError(f"NaN in scores ({np.isnan(scores).sum()} values)")
        metrics = evaluate(y, scores)
        metrics["time_s"] = round(time.time()-t0, 2)
        return metrics
    except Exception as e:
        print(f"    ✗ ERROR: {e}")
        return dict(auroc=0.5,aucpr=0.,f1=0.,precision=0.,recall=0.,time_s=0.)

print("run_method() and _VanillaAE defined")


## Cell 7 — Run all benchmarks

> Expected runtime: 5–20 min depending on machine.

**Note on fallback results:** If ADBench is not installed, the synthetic fallback produces data that is too easy. Expect all methods to score ~1.0 on most datasets. These results cannot be used for the paper — they are only for testing the pipeline.

In [ ]:
DATASETS = ["Thyroid","WBC","Annthyroid","Lympho",
            "Cardiotocography","Ionosphere","Arrhythmia","Satellite"]
METHODS  = ["Deep-MELU","Isolation Forest","LOF","OC-SVM","Vanilla AE"]

results = {}

for ds_name in DATASETS:
    print(f"\n{'='*55}")
    print(f"Dataset: {ds_name}")
    print(f"{'='*55}")
    X, y = load_dataset(ds_name)
    results[ds_name] = {}
    for method in METHODS:
        print(f"  {method:<22}", end="", flush=True)
        m = run_method(method, X, y)
        results[ds_name][method] = m
        ok = "✓" if m["auroc"] > 0.5 else "✗"
        print(f"  {ok}  AUROC={m['auroc']:.3f}  AUCPR={m['aucpr']:.3f}  "
              f"F1={m['f1']:.3f}  ({m['time_s']}s)")

print("\n✓ All benchmarks complete.")


## Cell 8 — Results table and plots

In [ ]:
rows = []
for ds in DATASETS:
    for method in METHODS:
        m = results[ds][method]
        rows.append({"Dataset":ds,"Method":method,
                     "AUROC":round(m["auroc"],4),"AUCPR":round(m["aucpr"],4),
                     "F1":round(m["f1"],4),"Precision":round(m["precision"],4),
                     "Recall":round(m["recall"],4),"Time(s)":m["time_s"]})
df = pd.DataFrame(rows)

pivot = df.pivot(index="Dataset",columns="Method",values="AUROC").round(3)
pivot["Best"]          = pivot.max(axis=1)
pivot["Deep-MELU rank"]= pivot[METHODS].rank(axis=1, ascending=False)["Deep-MELU"].astype(int)
display(pivot.style.background_gradient(cmap="YlGn",subset=METHODS).format("{:.3f}"))

# Plot
fig, axes = plt.subplots(2,2,figsize=(16,10))
fig.suptitle("Deep-MELU vs Baselines — ADBench Real Datasets (v2)", fontsize=14)
COLORS = {"Deep-MELU":"#1D9E75","Isolation Forest":"#534AB7",
          "LOF":"#BA7517","OC-SVM":"#888780","Vanilla AE":"#D85A30"}
x = np.arange(len(DATASETS)); w = 0.15
offsets = np.linspace(-(len(METHODS)-1)/2,(len(METHODS)-1)/2,len(METHODS))

for ax_i, metric in enumerate(["auroc","aucpr"]):
    ax = axes[0,ax_i]
    for i,method in enumerate(METHODS):
        vals = [results[d][method][metric] for d in DATASETS]
        ax.bar(x+offsets[i]*w, vals, width=w,
               label=method, color=COLORS[method], alpha=0.87)
    ax.axhline(0.5,color="gray",lw=0.8,ls="--",alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels([d[:8] for d in DATASETS],rotation=20,fontsize=9)
    ax.set_ylabel(metric.upper()); ax.set_ylim(0.3,1.05)
    ax.set_title(f"{metric.upper()} per dataset"); ax.legend(fontsize=8,ncol=2)
    ax.grid(axis="y",alpha=0.3)

ax = axes[1,0]
metric_names = ["auroc","aucpr","f1","precision","recall"]
for i,method in enumerate(METHODS):
    vals = [np.mean([results[d][method][m] for d in DATASETS]) for m in metric_names]
    ax.bar(np.arange(len(metric_names))+offsets[i]*0.15, vals, width=0.15,
           label=method, color=COLORS[method], alpha=0.87)
ax.set_xticks(range(len(metric_names))); ax.set_xticklabels(metric_names,fontsize=9)
ax.set_title("Average across all 8 datasets"); ax.legend(fontsize=8,ncol=2); ax.set_ylim(0,1.05)
ax.grid(axis="y",alpha=0.3)

ax = axes[1,1]
hm = np.array([[results[d][m]["auroc"] for m in METHODS] for d in DATASETS])
sns.heatmap(hm,annot=True,fmt=".3f",
            xticklabels=METHODS,yticklabels=[d[:10] for d in DATASETS],
            cmap="YlGn",vmin=0.4,vmax=1.0,ax=ax,annot_kws={"size":8})
ax.set_title("AUROC heatmap"); ax.set_xticklabels(METHODS,rotation=20,fontsize=8)

plt.tight_layout()
plt.savefig("outputs/adbench_results_v2.png",dpi=150,bbox_inches="tight")
plt.show()
print("Figure saved → outputs/adbench_results_v2.png")
df.to_csv("outputs/adbench_results_full.csv",index=False)
print("CSV saved → outputs/adbench_results_full.csv")


## Cell 9 — Summary and interpretation

In [ ]:
summary_rows = []
for method in METHODS:
    avg_auroc = np.mean([results[d][method]["auroc"] for d in DATASETS])
    avg_aucpr = np.mean([results[d][method]["aucpr"] for d in DATASETS])
    avg_f1    = np.mean([results[d][method]["f1"]    for d in DATASETS])
    nan_count = sum(results[d][method]["auroc"] == 0.5
                    and results[d][method]["f1"] == 0.0 for d in DATASETS)
    summary_rows.append({
        "Method":method, "Avg AUROC":round(avg_auroc,4),
        "Avg AUCPR":round(avg_aucpr,4), "Avg F1":round(avg_f1,4),
        "NaN/crash count":nan_count,
    })

df_sum = pd.DataFrame(summary_rows).sort_values("Avg AUROC",ascending=False)
df_sum.index = range(1, len(df_sum)+1); df_sum.index.name="Rank"
display(df_sum.style
        .bar(subset=["Avg AUROC"],color="#1D9E75",vmin=0.5,vmax=1.0)
        .format({"Avg AUROC":"{:.4f}","Avg AUCPR":"{:.4f}","Avg F1":"{:.4f}"})
        .set_caption("Summary — ADBench real datasets (v2)"))

df_sum.to_csv("outputs/adbench_results_summary.csv")

print()
print("=== Interpretation ===")
dm_auroc = [results[d]["Deep-MELU"]["auroc"] for d in DATASETS]
nan_ds   = [d for d in DATASETS if results[d]["Deep-MELU"]["auroc"] == 0.5
            and results[d]["Deep-MELU"]["f1"] == 0.0]
if nan_ds:
    print(f"⚠️  Deep-MELU still scored 0.500 on: {nan_ds}")
    print("   Check for NaN in score output.")
else:
    print("✓ No NaN crashes detected in v2.")
print()
print("Note: if using fallback simulation, all methods score ~1.0.")
print("Install ADBench for real dataset comparisons:")
print("  pip install adbench")
